In [ ]:
# libraries
!pip install transformers datasets evaluate

!pip install numpy==1.26.4 --force-reinstall
import numpy as np
print(np.__version__)


  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
1.26.4


In [ ]:
import os
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
import evaluate
import pandas as pd

In [ ]:
# Set seed for reproducibility
set_seed(42)

# Load raw .txt file
lines = open("drive/MyDrive/80000clear.txt", encoding="utf-8").readlines()
labels, texts = [], []

for line in lines:
    if '\t' in line:
        label, text = line.strip().split('\t', 1)
        labels.append(label)
        texts.append(text)

# Turn into Hugging Face Dataset
df = pd.DataFrame({"label": labels, "text": texts})
dataset = Dataset.from_pandas(df)

# Label mapping
label2id = {
    "Edu": 0,
    "Exp": 1,
    "Obj": 2,
    "PI": 3,
    "QC": 4,
    "Skill": 5,
    "Sum": 6
}
id2label = {v: k for k, v in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode_labels)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/78669 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Map:   0%|          | 0/78669 [00:00<?, ? examples/s]

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
from transformers import Trainer, TrainingArguments

# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Split manually (85/15)
split = int(0.85 * len(dataset))
train_dataset = dataset.select(range(split))
eval_dataset = dataset.select(range(split, len(dataset)))

# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=7,
    id2label=id2label,
    label2id=label2id
)

for param in model.distilbert.parameters():
    param.requires_grad = False

# THEN UNFREEZE SPECIFIC LAYERS
# Last 4 transformer layers + classifier
for layer in model.distilbert.transformer.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True  # Now only these update

# keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# 3. Set the model to the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training settings
training_args = TrainingArguments(
    output_dir="drive/MyDrive/Resume_sentence_classifier_V3",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.0,
    logging_dir="./logs",
    logging_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=["tensorboard"],
    seed=42
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-e0182aed9456>:70: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
!pip install tensorboard

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.479000,0.518166,0.849250,0.850702,0.849250,0.849028
2,0.395900,0.515417,0.850267,0.847319,0.850267,0.847832
3,0.366600,0.523708,0.851369,0.850598,0.851369,0.850104


TrainOutput(global_step=12540, training_loss=0.42901713753050785, metrics={'train_runtime': 7221.2918, 'train_samples_per_second': 27.78, 'train_steps_per_second': 1.737, 'total_flos': 2.6575859542339584e+16, 'train_loss': 0.42901713753050785, 'epoch': 3.0})